# Explaining a model and checking where it fails

Data Scientist track, lesson DS11. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** shuffle columns to see what a model uses, draw a partial dependence curve, compare error rates across groups with an interval, name a fairness risk, and write a model card and a short release checklist.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")


con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)


# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")


## The model and the split

The split of lessons DS2 to DS5 (13,362 people to fit, 4,454 to test) and the day-7 logistic regression of lesson DS4. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
ACTIVITY = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d"]       # four ways of counting how active a person was in week 1
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)


def logit(static=STATIC):
    """The day-7 logistic regression of lessons DS2 to DS5: one-hot signup columns, scaled week-1 columns."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), static), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))


def tree(depth, leaf=1):
    """The decision tree of lesson DS5."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", "passthrough", WEEK1)]),
                         DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=SEED))


model = logit().fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]
yt = y_test.to_numpy()
print(len(X_train), len(X_test), round(float(roc_auc_score(yt, p_test)), 3))

## Unit 1: Shuffle a column and see what the model loses

**Predict first.** `sessions_7d`, `expenses_7d`, `recent_3d` and `active_days_7d` all count how active a person was in week 1. Shuffled one at a time, the biggest loss is under 0.03. Shuffled together, will the loss be about the same as the biggest one, about the sum of the four, or more than the sum?

In [ ]:
def group_loss(m, cols, repeats=30):
    """Mean drop in ROC AUC on the test people when the rows of `cols` are shuffled together, as one block."""
    rng = np.random.default_rng(SEED)
    base = roc_auc_score(yt, m.predict_proba(X_test)[:, 1])
    drops = []
    for _ in range(repeats):
        shuffled = X_test.copy()
        shuffled[cols] = X_test[cols].iloc[rng.permutation(len(X_test))].to_numpy()
        drops.append(base - roc_auc_score(yt, m.predict_proba(shuffled)[:, 1]))
    return float(np.mean(drops))


one_by_one = permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=30, random_state=SEED)
loss = pd.Series(one_by_one.importances_mean, index=X_test.columns).sort_values(ascending=False)
print(loss.round(4).to_string())
print(round(float(loss[ACTIVITY].sum()), 4), round(group_loss(model, ACTIVITY), 4))       # the four activity columns: one by one added up, and shuffled together

## Unit 2: What the model says when one column moves

**Predict first.** At 0 expenses in week 1 the model's average chance of churning is about 0.60. Set every test person to 6 expenses and leave the other columns as they are: will the average be about 0.55, about 0.42 or about 0.20? Then compare it with the share who really churned among people who have 6 expenses.

In [ ]:
values = [0, 1, 2, 3, 4, 6]


def average_chance(col, values):
    """Partial dependence by hand: set the column to each value for every test person, and average the model's chance."""
    out = []
    for v in values:
        shifted = X_test.copy()
        shifted[col] = v
        out.append(float(model.predict_proba(shifted)[:, 1].mean()))
    return out


by_hand = average_chance("expenses_7d", values)
sk = partial_dependence(model, X_test.astype({"expenses_7d": float}), ["expenses_7d"], kind="average", grid_resolution=50)       # scikit-learn picks its own grid of values
agrees = float(np.abs(np.array(average_chance("expenses_7d", sk["grid_values"][0])) - sk["average"][0]).max()) < 1e-9
print(np.round(by_hand, 3), agrees)
print(np.round([float(y_test[X_test["expenses_7d"] == v].mean()) for v in values], 3))      # the share who really churned among the test people with that many expenses
print([int((X_test["expenses_7d"] == v).sum()) for v in values])

## Unit 3: Where it fails: the same model, group by group

**Predict first.** The rule offers the credit to 22.4 percent of all test people. Among the web people, will the share offered be about the same as among the iOS people, about half, or about a tenth?

In [ ]:
THRESHOLD = 2 / 3                    # lesson DS4's rule: an offer costs $2 and a missed leaver $1, so offer when the chance is at least 2 / (2 + 1)
flag = p_test >= THRESHOLD


def slice_table(col, threshold=THRESHOLD):
    """For each level of `col`: people, share who churned, mean chance, ROC AUC, share offered, share of the leavers offered (recall)."""
    flagged = p_test >= threshold
    rows = {}
    for level in sorted(X_test[col].unique()):
        m = (X_test[col] == level).to_numpy()
        rows[level] = {"people": int(m.sum()), "churned": round(float(yt[m].mean()), 3), "mean chance": round(float(p_test[m].mean()), 3),
                       "ROC AUC": round(float(roc_auc_score(yt[m], p_test[m])), 3), "offered": round(float(flagged[m].mean()), 3),
                       "recall": round(float(flagged[m & (yt == 1)].mean()), 3)}
    return pd.DataFrame(rows).T


print(slice_table("platform").to_string())
web = (X_test["platform"] == "web").to_numpy()
rng = np.random.default_rng(SEED)
gaps = []
for _ in range(2000):                              # resample the test people; recall of the app people minus recall of the web people
    i = rng.integers(0, len(yt), len(yt))
    f, yy, w = flag[i], yt[i], web[i]
    gaps.append(f[~w & (yy == 1)].mean() - f[w & (yy == 1)].mean())
print(round(float(flag[~web & (yt == 1)].mean() - flag[web & (yt == 1)].mean()), 3), np.round(np.percentile(gaps, [2.5, 97.5]), 3))

## Unit 4: Write the card and run the checklist

**Predict first.** After the card is written, which checklist items stay open, and may the model be released?

In [ ]:
rng = np.random.default_rng(SEED)
lo, hi = np.percentile([roc_auc_score(yt[i], p_test[i]) for i in (rng.integers(0, len(yt), len(yt)) for _ in range(2000))], [2.5, 97.5])      # 2,000 resamples of the test people
by_platform = slice_table("platform")
web_row, app_row = by_platform.loc["web"], by_platform.loc[["android", "ios"]]
card = {
    "Model details": "Logistic regression on week-1 columns; 13,362 training people; scikit-learn, seed 2026.",
    "Intended use": "Rank people by their chance of leaving in days 8 to 30, to decide who gets a win-back offer. Nothing else.",
    "Factors": "Channel, platform, region. The data has no protected attribute: a real card names the groups the model could affect.",
    "Metrics": f"ROC AUC {roc_auc_score(yt, p_test):.3f} (95% interval {lo:.3f} to {hi:.3f}); offer when the chance is at least {THRESHOLD:.3f}.",
    "Evaluation data": "4,454 held-out people who signed up 2026-01-01 to 2026-05-31, 30 days of follow-up each.",
    "Training data": "The other 13,362 people of the same table; 54.4% churners in both parts.",
    "Quantitative analysis": f"Share of leavers offered: {float(web_row['recall']):.3f} on web, {float(app_row['recall'].mean()):.3f} on iOS and Android.",
    "Ethical considerations": "The offer is a benefit, and web leavers get it far less often: web people rarely come from paid social.",
    "Caveats and recommendations": "The costs of the rule are assumptions. Recheck when the channel mix changes.",
}
print(" | ".join(card))
checklist = {"Owner and contact named": None, "Intended use and out-of-scope uses written": "Intended use", "Training and evaluation data described": "Evaluation data",
             "Metrics reported with an interval on held-out people": "Metrics", "Groups checked, gaps and causes written": "Quantitative analysis",
             "Who is affected and the possible harm written": "Ethical considerations", "Assumptions and limits written": "Caveats and recommendations",
             "A person can override, and people can appeal": None, "Monitoring, retraining trigger and rollback": None}
status = {item: "done" if section and card.get(section) else "open" for item, section in checklist.items()}
for item, s in status.items():
    print(f"[{'x' if s == 'done' else ' '}] {item}")
print("release:", "ready" if all(s == "done" for s in status.values()) else "not yet, open: " + str(sum(s == "open" for s in status.values())))

## Finish it

### Exercise 1: Finish it

**Task:** **The same shuffle on a tree.** The starter fits a depth-4 tree. Shuffle the signup columns (`STATIC`) together, then the week-1 columns (`WEEK1`) together, and read the drop in ROC AUC of this tree each time, using `group_loss` of unit 1. Put the list [drop for the signup columns, drop for the week-1 columns] in `answer_1`, each rounded to 3 decimals.

**Expected:** `answer_1` is a list of two numbers.

In [ ]:
tree4 = tree(4, 50).fit(X_train, y_train)      # given: a depth-4 tree with at least 50 people in every leaf
# shuffle the signup columns together, then the week-1 columns together, and read the drop in ROC AUC of this tree
answer_1 = None  # replace None with [drop for the signup columns, drop for the week-1 columns], each rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [0.094, 0.063], traps=[([0.101, 0.057], "Those are the numbers of the logistic model. Shuffle the columns for tree4.")], tol=2e-3)

<details>
<summary>Hint</summary>

`group_loss(m, cols)` takes the fitted model first. Pass `tree4` and the name of each group, and round each result.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = [round(group_loss(tree4, ...), 3), round(group_loss(tree4, ...), 3)]
```

</details>

<details>
<summary>Solution</summary>

```python
tree4 = tree(4, 50).fit(X_train, y_train)
answer_1 = [round(group_loss(tree4, STATIC), 3), round(group_loss(tree4, WEEK1), 3)]
```

On the test people, shuffling the signup columns together costs this tree 0.094 of ROC AUC and shuffling the week-1 columns 0.063. The logistic regression of unit 1 loses 0.101 and 0.057, so both models lean on how a person arrived first, and the same `group_loss` explained both without a line about trees.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Who is reached at another threshold?** Before you start, say in one sentence what you expect: a higher threshold offers fewer people, so fewer leavers of every platform are reached. Now a wasted offer costs $3 and a leaver left without one forgoes $1, so the rule offers when the chance is at least 3 / (3 + 1) = 0.75. Use `p_test` of unit 3 and put the list [share of the web leavers offered, share of the iOS and Android leavers offered, taken together] in `answer_2`, each rounded to 3 decimals. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
# write your code here

In [ ]:
_ = check("Exercise 2", "answer_2", [0.041, 0.369], traps=[([0.047, 0.421], "That is the threshold of unit 3, 2 / 3. Use 0.75.")], tol=2e-3)

<details>
<summary>Hint</summary>

Flag the people whose chance is at least 0.75, keep only the leavers (`yt == 1`), and take the mean of the flag for web and for the other two platforms.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Both shares fall, and the web leavers reached are a small fraction of the app leavers reached."*

```python
flag75 = p_test >= 0.75
answer_2 = [round(float(flag75[... & leavers].mean()), 3), round(float(flag75[... & leavers].mean()), 3)]
```

</details>

<details>
<summary>Solution</summary>

```python
flag75 = p_test >= 0.75
leavers = yt == 1
web_mask = (X_test["platform"] == "web").to_numpy()
answer_2 = [round(float(flag75[web_mask & leavers].mean()), 3), round(float(flag75[~web_mask & leavers].mean()), 3)]
```

At the threshold 0.75 the rule offers fewer people, and both shares fall: 0.041 of the web leavers and 0.369 of the app leavers. The web leavers are still reached at about a ninth of the rate of the app leavers, as at 2 / 3 (0.047 against 0.421): a threshold changes how many people are reached, not who is left out.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague refits the model without the `platform` column, "so it cannot treat platforms differently", and sees no problem. Measure it: put in `answer_3` the list [share of the web test people offered, share of the iOS and Android test people offered, taken together] that the model without `platform` gives at the threshold 0.6, each rounded to 3 decimals.

**Expected:** `answer_3` is a list of two numbers between 0 and 1.

In [ ]:
unaware = logit(["channel", "region"]).fit(X_train.drop(columns="platform"), y_train)      # colleague: "the model never sees platform, so it cannot treat platforms differently"
p_unaware = unaware.predict_proba(X_test.drop(columns="platform"))[:, 1]
answer_3 = None  # replace None with [share of web test people offered, share of iOS and Android test people offered] at the threshold 0.6, each rounded to 3 decimals

In [ ]:
_ = check("Exercise 3", "answer_3", [0.071, 0.434], tol=2e-3)

<details>
<summary>Hint</summary>

`p_unaware` holds the chances of the model that never sees `platform`. Make a mask of the web rows, take the share of `p_unaware` at or above 0.6 inside it, and again outside it.

</details>

<details>
<summary>Half-solution</summary>

```python
is_web = (X_test["platform"] == "web").to_numpy()
answer_3 = [round(float((p_unaware[is_web] >= 0.6).mean()), 3), round(float((p_unaware[...] >= 0.6).mean()), 3)]
```

</details>

<details>
<summary>Solution</summary>

```python
unaware = logit(["channel", "region"]).fit(X_train.drop(columns="platform"), y_train)
p_unaware = unaware.predict_proba(X_test.drop(columns="platform"))[:, 1]
is_web = (X_test["platform"] == "web").to_numpy()
answer_3 = [round(float((p_unaware[is_web] >= 0.6).mean()), 3), round(float((p_unaware[~is_web] >= 0.6).mean()), 3)]
```

Without `platform` the model offers 0.071 of the web people at 0.6 and 0.434 of the app people, and its ROC AUC is 0.688 against 0.690. The colleague checked the inputs, but fairness is judged on the outputs. Hardt, Price and Srebro call fairness through unawareness ineffective because of redundant encodings, ways of predicting a group from other columns: here the channel carries it, since no app-store person is on the web and only 5 percent of paid-social people are.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")